# Strands Agents 入门

本实验介绍 Strands Agents，一个用于构建具有工具集成能力的 AI 智能体的强大框架。您将学习创建智能体、使用内置工具和开发自定义工具的基础知识。

## 概述

在本实验中，您将：
- 了解 Strands Agents 的核心概念
- 使用内置工具创建您的第一个 AI 智能体
- 为特定用例构建自定义工具
- 探索对话历史和智能体记忆
- 学习智能体开发的最佳实践

## 前提条件

在开始本实验之前，请确保您已具备：
- 已配置 AWS 凭证（IAM 角色或环境变量）
- 已安装所需的 Python 软件包
- 基于 AWS 区域的 Nova Pro 模型 ID

如果您不是在已承担 IAM 角色的环境中运行，请将 AWS 凭证设置为环境变量：

In [ ]:
import os

#os.environ["AWS_ACCESS_KEY_ID"]=<YOUR ACCESS KEY>
#os.environ["AWS_SECRET_ACCESS_KEY"]=<YOUR SECRET KEY>
#os.environ["AWS_SESSION_TOKEN"]=<OPTIONAL - YOUR SESSION TOKEN IF TEMP CREDENTIAL>
#os.environ["AWS_REGION"]=<AWS REGION WITH BEDROCK AGENTCORE AVAILABLE>

安装 Strands Agents 所需的软件包：

In [ ]:
#%pip install -q strands-agents strands-agents-tools rich

根据 AWS 区域设置 Nova Pro 模型 ID：

In [ ]:
import boto3

region = boto3.session.Session().region_name

NOVA_PRO_MODEL_ID = "us.amazon.nova-pro-v1:0"
if region.startswith("eu"):
    NOVA_PRO_MODEL_ID = "eu.amazon.nova-pro-v1:0"
elif region.startswith("ap"):
    NOVA_PRO_MODEL_ID = "apac.amazon.nova-pro-v1:0"

print(f"Nova Pro Model ID: {NOVA_PRO_MODEL_ID}")

## 什么是 Strands Agents？

Strands Agents 是一个 Python 框架，简化了具有工具集成能力的 AI 智能体的创建过程。主要特性包括：

- **简单的智能体创建**：易于使用的 API，用最少的代码创建 AI 智能体
- **内置工具**：预构建的工具，如计算器、网络搜索等
- **自定义工具支持**：使用简单的 Python 函数创建您自己的工具
- **对话记忆**：自动对话历史管理
- **模型灵活性**：支持各种语言模型，包括 Amazon Bedrock 和 OpenAI 中的模型

Strands Agents 为构建能够与外部系统交互并执行复杂任务的高级 AI 应用程序提供了基础。

## 创建您的第一个 Strands 智能体

让我们从创建一个带有内置计算器工具的简单智能体开始。这演示了 Strands Agents 的基本结构以及工具是如何集成的：

In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel
from strands_tools import calculator

# Create your first agent
agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID),
    system_prompt="你是一个生活助手，运用科学的知识回答各种问题。",
    tools=[calculator],
)

agent("半径为 8.26 厘米的圆的面积是多少？")

## 构建自定义工具

Strands Agents 的一个强大特性是能够创建自定义工具。让我们创建一个天气工具并将其与内置计算器结合使用：

In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel
from strands_tools import calculator

# 创建一个用于演示的自定义天气工具
@tool
def weather(city: str) -> str:
    """获取城市的天气信息
    Args:
        city: 城市或地点名称
    """
    return f"{city}的天气：晴天，35°C"  # 用于演示的虚拟结果

# Create your first agent
agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID),
    system_prompt="你是一个生活助手，运用科学的知识回答各种问题。",
    tools=[weather, calculator],
)

agent("香港的天气怎么样？请用华氏度返回温度。")

## 理解智能体执行流程

让我们来看看 Strands Agents 如何通过智能体循环处理请求。这有助于理解智能体框架的内部工作原理：

In [ ]:
from rich.table import Table
import rich
import json

console = rich.get_console()

console.print("Agent Loop Detail")
console.rule()
console.print(f"Number of Loops: {agent.event_loop_metrics.cycle_count}")

table = Table(title="Agent Messages", show_lines=True)
table.add_column("Role", style="green")
table.add_column("Text", style="magenta")
table.add_column("Tool Name", style="cyan")
table.add_column("Tool Input", style="cyan")
table.add_column("Tool Result", style="cyan")

for message in agent.messages:
    text = [content["text"] for content in message["content"] if "text" in content]
    tool_name = [content["toolUse"]["name"] for content in message["content"] if "toolUse" in content]
    tool_input = [content["toolUse"]["input"] for content in message["content"] if "toolUse" in content]
    tool_result = [content["toolResult"]["content"][0] for content in message["content"] if "toolResult" in content]
    table.add_row(message["role"], text[-1] if text else "", 
                  tool_name[-1] if tool_name else "", 
                  json.dumps(tool_input[-1], indent=2) if tool_input else "", 
                  (json.dumps(tool_result[-1], indent=2)[:500]+"\n.\n.\n." if len(str(tool_result[-1])) > 500 else json.dumps(tool_result[-1], indent=2)) if tool_result else "")

console.print(table)

## 智能体循环

智能体循环是 Strands Agents SDK 中的核心概念，它通过推理、工具使用和响应生成的循环来实现智能的自主行为。

![strands-agents-agent-loop](images/strands-agents-agent-loop.png)

其核心步骤如下：

1. 接收用户输入和上下文信息
2. 使用语言模型（LLM）处理输入
3. 决定是否使用工具来收集信息或执行操作
4. 执行工具并接收结果
5. 利用新信息继续推理
6. 生成最终响应或再次迭代循环
   
在单次用户交互中，此循环可能重复多次，使智能体能够执行复杂的多步推理和自主行为。

参考：[Strands Agents - Agent Loop](https://strandsagents.com/latest/documentation/docs/user-guide/concepts/agents/agent-loop/)


## Strands Agents 对话管理

Strands Agents 包含内置的对话管理功能，默认使用 `SlidingWindowConversationManager` 策略。该系统处理上下文管理、内存优化和对话流程，确保智能体能够在遵守模型上下文限制的同时维持连贯的长期交互。它会在会话内自动维护对话上下文。

**参考：** [Strands Agents - Conversation Management](https://strandsagents.com/latest/documentation/docs/user-guide/concepts/agents/conversation-management/)

In [ ]:
agent("我们之前聊了什么？")

让我们来看看 Strands Agents 如何管理对话历史。这有助于理解智能体框架的内部工作原理：

In [ ]:
from rich.table import Table
import rich
import json

console = rich.get_console()

console.print("Agent Loop Detail")
console.rule()
console.print(f"Number of Loops: {agent.event_loop_metrics.cycle_count}")

table = Table(title="Agent Messages", show_lines=True)
table.add_column("Role", style="green")
table.add_column("Text", style="magenta")
table.add_column("Tool Name", style="cyan")
table.add_column("Tool Input", style="cyan")
table.add_column("Tool Result", style="cyan")

for message in agent.messages:
    text = [content["text"] for content in message["content"] if "text" in content]
    tool_name = [content["toolUse"]["name"] for content in message["content"] if "toolUse" in content]
    tool_input = [content["toolUse"]["input"] for content in message["content"] if "toolUse" in content]
    tool_result = [content["toolResult"]["content"][0] for content in message["content"] if "toolResult" in content]
    table.add_row(message["role"], text[-1] if text else "", 
                  tool_name[-1] if tool_name else "", 
                  json.dumps(tool_input[-1], indent=2) if tool_input else "", 
                  (json.dumps(tool_result[-1], indent=2)[:500]+"\n.\n.\n." if len(str(tool_result[-1])) > 500 else json.dumps(tool_result[-1], indent=2)) if tool_result else "")

console.print(table)

### 在新会话中检查 Strands Agents 对话管理

Strands Agents 对话管理通过在会话内的输入提示中累积用户和智能体消息来维护对话历史。对话历史存储在内存中，当会话结束时会丢失。让我们通过初始化一个新的智能体来模拟新会话，并测试 Strands Agents 对话管理的行为。

In [ ]:
# Initialise a new agent for a new session
agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID),
    system_prompt="你是一个生活助手，运用科学的知识回答各种问题。",
    tools=[weather, calculator],
)

agent("我们之前聊了什么？")

 您会看到智能体忘记了之前关于香港天气查询的对话。这说明 Strands Agents 内置的对话管理仅支持短期记忆。要实现跨会话和跨智能体的长期记忆，您将在后续的 [实验 6：Strands Agents 与 Bedrock AgentCore Memory](../06-bedrock-agentcore-memory/06-agentcore-memory.ipynb) 中探索 Bedrock AgentCore Memory 作为托管智能体记忆服务。

## 总结

在本 Strands Agents 入门实验中，您学习了：

### 我们完成了什么
- **创建了您的第一个 AI 智能体**，使用内置工具（计算器）
- **构建了自定义工具**，使用 `@tool` 装饰器实现特定用例
- **探索了对话管理**，了解不同的 ConversationManager 类型
- **检查了智能体执行流程**和消息历史跟踪

### Strands 的其他功能
除了我们已介绍的内容，Strands Agents 还提供：

- **Model Context Protocol (MCP) 支持**：与 MCP 服务器集成以扩展工具能力
- **多智能体模式**：协调多个智能体处理复杂工作流
- **会话管理**：持久化对话存储和检索
- **流式响应**：实时响应生成，提升用户体验
- **自定义模型集成**：支持 Amazon Bedrock 之外的各种 LLM 提供商

### 后续步骤：Bedrock AgentCore 集成

现在您已经了解了 Strands Agents 的基础知识，我们将探索如何通过与 **Amazon Bedrock AgentCore** 集成来增强这些功能。此集成提供：

- **代码解释器**：在智能体中动态执行 Python 代码
- **浏览器自动化**：网页交互和数据提取能力
- **安全凭证管理**：安全处理外部 API 密钥和密钥
- **运行时部署**：在云环境中可扩展的智能体部署
- **增强的可观测性**：用于生产环境智能体的监控和调试工具